<a href="https://colab.research.google.com/github/nishantsakesh/TEXT-AND-VIDEO-GENERATION-PIPELINE-IN-GOOGLE-COLAB/blob/main/Text_to_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate opencv-python Pillow segment-anything
import numpy as np
!git clone https://github.com/megvii-research/ECCV2022-RIFE.git
%cd ECCV2022-RIFE
!pip install -r requirements.txt
!pip install gdown
!gdown --id 1APIzVeI-4ZZCEuIRE1m6WYfSCaOsi_7_
!unzip /content/ECCV2022-RIFE/RIFE_trained_model_v3.6.zip -d /content/ECCV2022-RIFE

fatal: destination path 'ECCV2022-RIFE' already exists and is not an empty directory.
/content/ECCV2022-RIFE
  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
pymc 5.21.2 requires numpy>=1.25.0, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.23.5 which is incompatible.
tensorflo

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Traceback (most recent call last):
  File "/usr/lib/python3.11/http/client.py", line 1395, in getresponse
^C
Archive:  /content/ECCV2022-RIFE/RIFE_trained_model_v3.6.zip
replace /content/ECCV2022-RIFE/train_log/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install -U numpy --ignore-installed

  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.


In [ ]:
!pip install --upgrade "jax[cpu]"  # or "jax[cuda]" if you have a CUDA-enabled GPU
!pip install --upgrade jaxlib  # or "jaxlib==0.4.13+cuda12.cudnn89" for CUDA 12 and cuDNN 8.9

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.8 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.5.1
    Uninstalling jaxlib-0.5.1:
      Successfully uninstalled jaxlib-0.5.1
  Attempting uninstall: jax
    Found existing installation: jax 0.5.2
    Uninstalling jax-0.5.2:
      Successfully uninstalled jax-0.5.2


^C


In [ ]:
!pip install transformers

In [ ]:
import numpy as np
import cv2
import torch
import jax
import jax.numpy as jnp
from diffusers import StableDiffusionImg2ImgPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import os
import urllib.request
from segment_anything import SamPredictor, sam_model_registry
from transformers import CLIPProcessor, CLIPModel
import gc
import logging
import random
import time
import re
from transformers import CLIPTokenizer, pipeline, AutoTokenizer

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def download_sam_checkpoint(checkpoint_path="sam_vit_h_4b8939.pth"):
    sam_checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    if not os.path.exists(checkpoint_path):
        logging.info("Downloading SAM model checkpoint...")
        urllib.request.urlretrieve(sam_checkpoint_url, checkpoint_path)
        logging.info("Download complete!")

def load_sam_model(model_type="vit_h", checkpoint="sam_vit_h_4b8939.pth"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sam = sam_model_registry[model_type](checkpoint=checkpoint)
    sam.to(device=device)
    predictor = SamPredictor(sam)
    return predictor

def extract_foreground_with_sam(image, predictor, point_coords=None):
    image_np = np.array(image)
    predictor.set_image(image_np)
    if point_coords is None:
        point_coords = np.array([[image.width // 2, image.height // 2]])
    masks, _, _ = predictor.predict(point_coords=point_coords, point_labels=np.array([1]))
    mask = masks[0]
    return mask

def transform_mask(mask, scale_x=1.0, scale_y=1.0, rotate_angle=0, skew_x=0, skew_y=0):
    mask_image = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")
    original_width, original_height = mask_image.size

    new_width = int(original_width * scale_x)
    new_height = int(original_height * scale_y)
    mask_image = mask_image.resize((new_width, new_height), Image.Resampling.BILINEAR)

    if rotate_angle != 0:
        mask_image = mask_image.rotate(rotate_angle, resample=Image.Resampling.BILINEAR)

    if skew_x != 0 or skew_y != 0:
        mask_image = mask_image.transform(mask_image.size, Image.AFFINE, (1, skew_x, 0, skew_y, 1, 0), resample=Image.Resampling.BILINEAR)

    padded_mask = Image.new("L", (original_width, original_height), 0)
    padded_mask.paste(mask_image, ((original_width - mask_image.width) // 2, (original_height - mask_image.height) // 2))

    return np.array(padded_mask) / 255.0

def blend_foreground_background(foreground, background, mask):
    foreground = foreground.convert("RGBA")
    background = background.convert("RGBA")
    mask = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")
    blended = Image.composite(foreground, background, mask)
    return blended.convert("RGB")

def transform_image(image, prompt, img2img_pipe, strength=0.45):
    with torch.inference_mode():
        transformed_image = img2img_pipe(prompt=prompt, image=image, strength=strength).images[0]
    return transformed_image

def create_video(image_folder, output_video, fps=5):
    images = [img for img in os.listdir(image_folder) if img.endswith(".png")]
    images.sort(key=lambda x: int(x.split("_")[1].split(".")[0]))
    if not images:
        logging.error(f"No images found in {image_folder}")
        return
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = frame.shape
    video = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    for image in images:
        video.write(cv2.imread(os.path.join(image_folder, image)))
    cv2.destroyAllWindows()
    video.release()

def validate_generated_frame(image, prompt, clip_model, clip_processor):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = clip_processor(text=[prompt], images=image, return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs).logits_per_image.item()
    return outputs

from transformers import pipeline, AutoTokenizer

def generate_story_prompts(initial_prompt, num_frames, desired_action="evolving scene"):
    generator = pipeline('text-generation', model='gpt2')
    tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    prompts = [initial_prompt]
    current_prompt = initial_prompt

    instruction = f"Generate the next step in a visual story based on this image description: '{initial_prompt}'. The scene should be {desired_action} gradually over {num_frames - 1} steps. Describe the next subtle visual change or action in one short sentence."

    for i in range(num_frames - 1):
        prompt_input = f"{instruction} Previous step: '{current_prompt}'. Next step:"
        try:
            generated_sequences = generator(
                prompt_input,
                max_length=120,
                num_return_sequences=1,
                pad_token_id=generator.tokenizer.eos_token_id,
                temperature=0.8,
                top_p=0.9,
                do_sample=True, # Enable sampling for more varied output
            )

            if generated_sequences:
                next_sentence = generated_sequences[0]['generated_text'].split("Next step:")[-1].strip()
                if next_sentence and len(tokenizer.tokenize(next_sentence)) > 3:
                    potential_next_prompt = f"{current_prompt}, {next_sentence}"
                    if len(tokenizer.tokenize(potential_next_prompt)) <= 72:
                        prompts.append(potential_next_prompt)
                        current_prompt = potential_next_prompt
                        print(f"Prompt {i+2}: {current_prompt}")
                    else:
                        shortened_tokens = tokenizer.tokenize(next_sentence)[:(72 - len(tokenizer.tokenize(current_prompt)) - 2)]
                        if shortened_tokens:
                            shortened_sentence = tokenizer.decode(shortened_tokens).strip()
                            potential_next_prompt = f"{current_prompt}, {shortened_sentence}"
                            prompts.append(potential_next_prompt)
                            current_prompt = potential_next_prompt
                            print(f"Prompt {i+2}: {current_prompt}")
                        else:
                            prompts.append(current_prompt + ", the scene continues.")
                            print(f"Prompt {i+2}: {current_prompt + ', the scene continues.'}")
                else:
                    prompts.append(current_prompt + ", the visuals shift slightly.")
                    current_prompt += ", the visuals shift slightly."
                    print(f"Prompt {i+2}: {current_prompt}")
            else:
                prompts.append(current_prompt + ", more details emerge.")
                current_prompt += ", more details emerge."
                print(f"Prompt {i+2}: {current_prompt}")

        except Exception as e:
            logging.error(f"Error during GPT-2 prompt generation: {e}")
            prompts.append(current_prompt + ", something changes.")
            current_prompt += ", something changes."
            print(f"Prompt {i+2}: {current_prompt}")

    return prompts

def generate_story_video(image_path, initial_prompt, target_resolution, sam_predictor, clip_model, clip_processor, fps, video_length, strength=0.45, output_video="/content/story_video.mp4"):
    """
    Generates a story video from an input image and prompt, saving the video to /content/.
    """
    logging.info("Starting generate_story_video function...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        if not os.path.exists(image_path):
            logging.error(f"Image path not found: {image_path}")
            return

        original_image = Image.open(image_path).convert("RGB")
        resized_image = original_image.resize(target_resolution, Image.Resampling.LANCZOS)

        similarity = validate_generated_frame(resized_image, initial_prompt, clip_model, clip_processor)
        print(f"Initial similarity: {similarity}")
        if similarity < 0.3:
            logging.error(f"Image and text prompt are not very similar, similarity: {similarity}")
            return

        num_frames = int(fps * video_length)
        story_prompts = generate_story_prompts(initial_prompt, num_frames, desired_action="evolves") # More general desired action

        if torch.cuda.is_available():
            img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to(device)
            img2img_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(img2img_pipe.scheduler.config)
        else:
            img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5").to(device)
            img2img_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(img2img_pipe.scheduler.config)

        output_folder = "/content/story_frames"
        os.makedirs(output_folder, exist_ok=True)
        print(f"Output folder created at: {output_folder}")

        previous_frame = resized_image
        object_mask = extract_foreground_with_sam(previous_frame, sam_predictor)

        for i, prompt in enumerate(story_prompts):
            logging.info(f"Generating frame {i + 1} with prompt: {prompt}")
            start_time = time.time()
            transformed_frame = transform_image(previous_frame, prompt, img2img_pipe, strength)
            end_time = time.time()
            generation_time = end_time - start_time

            similarity = validate_generated_frame(transformed_frame, prompt, clip_model, clip_processor)
            if similarity < 0.25:
                logging.warning(f"Frame {i + 1} rejected due to low similarity ({similarity:.2f}). Retrying...")
                continue

            scale_x = random.uniform(0.95, 1.05)
            scale_y = random.uniform(0.95, 1.05)
            rotate_angle = random.uniform(-5, 5)
            transformed_mask = transform_mask(object_mask, scale_x, scale_y, rotate_angle)

            masked_frame = blend_foreground_background(transformed_frame, previous_frame, transformed_mask)
            masked_frame.save(os.path.join(output_folder, f"frame_{i:04d}.png"))
            logging.info(f"Saved frame {i + 1}/{len(story_prompts)} with similarity {similarity:.2f} and generation time: {generation_time:.2f} seconds")

            previous_frame = transformed_frame

            del transformed_frame
            gc.collect()
            torch.cuda.empty_cache()
            if i >= len(story_prompts) - 1:
                break

        create_video(output_folder, output_video, fps)
        logging.info("Story video generation complete!")

    except Exception as e:
        logging.error(f"Error generating story video: {e}")

if __name__ == "__main__":
    download_sam_checkpoint()
    image_path = input("Enter the path to your image: ")
    initial_prompt = input("Enter the initial text prompt: ")

    # Refine the initial prompt (example)
    initial_prompt = f"A vivid, detailed, and family-friendly image of {initial_prompt}."

    target_resolution = (512, 512)
    fps = int(input("Enter the frames per second (FPS): "))
    video_length = int(input("Enter the video length in seconds: "))

    sam_predictor = load_sam_model()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    output_folder = "/content/story_frames"
    output_video = "/content/story_video.mp4"

    # Delete existing files and folders
    if os.path.exists(output_folder):
        import shutil
        shutil.rmtree(output_folder)
    if os.path.exists(output_video):
        os.remove(output_video)

    try:
        generate_story_video(image_path, initial_prompt, target_resolution, sam_predictor, clip_model, clip_processor, fps, video_length, strength=0.3)
    except Exception as e:
        logging.error(f"An unexpected error occurred: {e}")

Enter the path to your image: /content/bird.jpg
Enter the initial text prompt: a bird sitting on the branch , opens its wings and starts flying into the sky
Enter the frames per second (FPS): 15
Enter the video length in seconds: 2
Initial similarity: 30.041725158691406


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
ERROR:root:Error during GPT-2 prompt generation: Input length of input_ids is 120, but `max_length` is set to 120. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.
ERROR:root:Error during GPT-2 prompt generation: Input length of input_ids is 120, but `max_length` is set to 120. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.
ERROR:root:Error during GPT-2 prompt generation: Input length of input_ids is 120, but `max_length` is set to 120. This can 

Prompt 2: A vivid, detailed, and family-friendly image of a bird sitting on the branch , opens its wings and starts flying into the sky., 'A vivid, detailed, and family-friendly image of a bird sitting
Prompt 3: A vivid, detailed, and family-friendly image of a bird sitting on the branch , opens its wings and starts flying into the sky., 'A vivid, detailed, and family-friendly image of a bird sitting, something changes.
Prompt 4: A vivid, detailed, and family-friendly image of a bird sitting on the branch , opens its wings and starts flying into the sky., 'A vivid, detailed, and family-friendly image of a bird sitting, something changes., something changes.
Prompt 5: A vivid, detailed, and family-friendly image of a bird sitting on the branch , opens its wings and starts flying into the sky., 'A vivid, detailed, and family-friendly image of a bird sitting, something changes., something changes., something changes.
Prompt 6: A vivid, detailed, and family-friendly image of a bird sitting

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Output folder created at: /content/story_frames


...

  0%|          | 0/15 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes .']


  0%|          | 0/15 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes ., something changes .']


  0%|          | 0/15 [00:00<?, ?it/s]